In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('covid_19_indonesia_time_series_all.csv')

In [4]:
print(df.shape)

(31822, 38)


In [6]:
print(df.columns.tolist())

['Date', 'Location ISO Code', 'Location', 'New Cases', 'New Deaths', 'New Recovered', 'New Active Cases', 'Total Cases', 'Total Deaths', 'Total Recovered', 'Total Active Cases', 'Location Level', 'City or Regency', 'Province', 'Country', 'Continent', 'Island', 'Time Zone', 'Special Status', 'Total Regencies', 'Total Cities', 'Total Districts', 'Total Urban Villages', 'Total Rural Villages', 'Area (km2)', 'Population', 'Population Density', 'Longitude', 'Latitude', 'New Cases per Million', 'Total Cases per Million', 'New Deaths per Million', 'Total Deaths per Million', 'Total Deaths per 100rb', 'Case Fatality Rate', 'Case Recovered Rate', 'Growth Factor of New Cases', 'Growth Factor of New Deaths']


In [7]:
print(df.dtypes)

Date                            object
Location ISO Code               object
Location                        object
New Cases                        int64
New Deaths                       int64
New Recovered                    int64
New Active Cases                 int64
Total Cases                      int64
Total Deaths                     int64
Total Recovered                  int64
Total Active Cases               int64
Location Level                  object
City or Regency                float64
Province                        object
Country                         object
Continent                       object
Island                          object
Time Zone                       object
Special Status                  object
Total Regencies                  int64
Total Cities                   float64
Total Districts                  int64
Total Urban Villages           float64
Total Rural Villages           float64
Area (km2)                       int64
Population               

In [8]:
print(df.head(3))

       Date Location ISO Code     Location  New Cases  New Deaths  \
0  3/1/2020             ID-JK  DKI Jakarta          2           0   
1  3/2/2020             ID-JK  DKI Jakarta          2           0   
2  3/2/2020               IDN    Indonesia          2           0   

   New Recovered  New Active Cases  Total Cases  Total Deaths  \
0              0                 2           39            20   
1              0                 2           41            20   
2              0                 2            2             0   

   Total Recovered  ...  Latitude New Cases per Million  \
0               75  ... -6.204699                  0.18   
1               75  ... -6.204699                  0.18   
2                0  ... -0.789275                  0.01   

   Total Cases per Million New Deaths per Million Total Deaths per Million  \
0                     3.60                    0.0                     1.84   
1                     3.78                    0.0                    

In [10]:
df['Date'] = pd.to_datetime(df['Date'])

In [11]:
df = df[df['Location Level'] == 'Province'].copy()
print(f"After province filter: {df.shape[0]:,} rows")

After province filter: 30,893 rows


In [12]:
# Clean percentage columns

df['Case Fatality Rate'] = (
    df['Case Fatality Rate']
    .str.replace('%', '', regex=False)
    .astype(float)
)

df['Case Recovered Rate'] = (
    df['Case Recovered Rate']
    .str.replace('%', '', regex=False)
    .astype(float)
)

In [13]:
# Monthly aggregation helper
df['Year Month'] = df['Date'].dt.to_period('M').astype(str)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()

In [14]:
# Active case ratio (how much of total cases are still active)
df['Active Case Ratio'] = (
    df['Total Active Cases'] / df['Total Cases'].replace(0, pd.NA)
).fillna(0).round(4)

In [15]:
# Handle nulls

df['Population'] = df['Population'].fillna(0).astype(int)
df['Population Density'] = df['Population Density'].fillna(0)
df['Area (km2)'] = df['Area (km2)'].fillna(0).astype(int)

In [16]:
# Tableau's built-in Indonesia map uses specific province name spellings
province_name_map = {
    'DKI Jakarta'                    : 'Jakarta',
    'DI Yogyakarta'                  : 'Yogyakarta',
    'Kepulauan Bangka Belitung'       : 'Bangka Belitung Islands',
    'Kepulauan Riau'                  : 'Riau Islands',
    'Nusa Tenggara Barat'            : 'West Nusa Tenggara',
    'Nusa Tenggara Timur'            : 'East Nusa Tenggara',
    'Kalimantan Barat'               : 'West Kalimantan',
    'Kalimantan Tengah'              : 'Central Kalimantan',
    'Kalimantan Selatan'             : 'South Kalimantan',
    'Kalimantan Timur'               : 'East Kalimantan',
    'Kalimantan Utara'               : 'North Kalimantan',
    'Sulawesi Utara'                 : 'North Sulawesi',
    'Sulawesi Tengah'                : 'Central Sulawesi',
    'Sulawesi Selatan'               : 'South Sulawesi',
    'Sulawesi Tenggara'              : 'Southeast Sulawesi',
    'Sulawesi Barat'                 : 'West Sulawesi',
    'Sumatera Utara'                 : 'North Sumatra',
    'Sumatera Barat'                 : 'West Sumatra',
    'Sumatera Selatan'               : 'South Sumatra',
    'Jawa Barat'                     : 'West Java',
    'Jawa Tengah'                    : 'Central Java',
    'Jawa Timur'                     : 'East Java',
    'Papua Barat'                    : 'West Papua',
    'Maluku Utara'                   : 'North Maluku',
}
df['Province Tableau'] = df['Province'].replace(province_name_map)

In [17]:
cols = [
    'Date', 'Year Month', 'Year', 'Month',
    'Province', 'Province Tableau', 'Island',
    'Latitude', 'Longitude',
    'New Cases', 'New Deaths', 'New Recovered', 'New Active Cases',
    'Total Cases', 'Total Deaths', 'Total Recovered', 'Total Active Cases',
    'New Cases per Million', 'Total Cases per Million',
    'Total Deaths per Million', 'Total Deaths per 100rb',
    'Case Fatality Rate', 'Case Recovered Rate',
    'Active Case Ratio',
    'Growth Factor of New Cases', 'Growth Factor of New Deaths',
    'Population', 'Population Density', 'Area (km2)',
]
df = df[cols]

In [18]:
print(f"\nFinal shape: {df.shape}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Provinces: {df['Province'].nunique()}")
print(f"Nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")


Final shape: (30893, 29)
Date range: 2020-03-01 → 2022-09-15
Provinces: 34
Nulls:
Growth Factor of New Cases     1935
Growth Factor of New Deaths    3442
dtype: int64


In [19]:
df.to_csv('covid_indonesia_clean.csv', index=False)
print("\nSaved: covid_indonesia_clean.csv")


Saved: covid_indonesia_clean.csv
